# Credit Risk Analysis

This notebook demonstrates the complete credit risk modeling pipeline:
1. Data generation
2. Exploratory data analysis
3. Model training
4. Model evaluation
5. Feature importance analysis

In [ ]:
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generators.credit_risk_generator import CreditRiskDataGenerator
from src.credit_risk.model import CreditRiskModel

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Generate Synthetic Data

In [ ]:
# Generate data
generator = CreditRiskDataGenerator(random_state=42)
df = generator.generate(n_samples=10000, default_rate=0.15)

print(f"Dataset shape: {df.shape}")
print(f"Default rate: {df['default'].mean():.2%}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Default rate by categorical features
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Home ownership
df.groupby('home_ownership')['default'].mean().sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue'
)
axes[0].set_title('Default Rate by Home Ownership')
axes[0].set_xlabel('Default Rate')

# Loan purpose
df.groupby('loan_purpose')['default'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='coral'
)
axes[1].set_title('Default Rate by Loan Purpose')
axes[1].set_xlabel('Default Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key features by default status
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

features = ['credit_score', 'annual_income', 'debt_to_income', 'loan_amount', 'age', 'interest_rate']

for idx, feature in enumerate(features):
    df[df['default'] == 0][feature].hist(bins=30, alpha=0.6, label='No Default', ax=axes[idx], color='green')
    df[df['default'] == 1][feature].hist(bins=30, alpha=0.6, label='Default', ax=axes[idx], color='red')
    axes[idx].set_title(f'{feature.replace("_", " ").title()} Distribution')
    axes[idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Model Training

In [ ]:
# Train XGBoost model
model = CreditRiskModel(model_type="xgboost", random_state=42)
metrics = model.train(df, test_size=0.2, cv_folds=5)

print("Model Performance Metrics:")
print("=" * 50)
for metric, value in metrics.items():
    if metric not in ['confusion_matrix', 'classification_report']:
        print(f"{metric}: {value:.4f}")

## 4. Model Evaluation

In [ ]:
# Confusion matrix visualization
from sklearn.metrics import ConfusionMatrixDisplay

cm = np.array(metrics['confusion_matrix'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Default', 'Default'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Classification report
print("Classification Report:")
print(metrics['classification_report'])

## 5. Feature Importance Analysis

In [ ]:
# Get feature importance
feature_importance = model.get_feature_importance()

# Plot
plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Feature Importance for Credit Risk Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Model Predictions

In [ ]:
# Make predictions on new data
new_data = generator.generate(n_samples=100, default_rate=0.15)
predictions = model.predict(new_data)
probabilities = model.predict_proba(new_data)

# Add predictions to dataframe
new_data['predicted_default'] = predictions
new_data['default_probability'] = probabilities

print("Sample predictions:")
print(new_data[['credit_score', 'annual_income', 'debt_to_income', 'default', 'predicted_default', 'default_probability']].head(10))

In [ ]:
# Distribution of predicted probabilities
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# All predictions
axes[0].hist(new_data['default_probability'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Default Probabilities')
axes[0].set_xlabel('Default Probability')
axes[0].set_ylabel('Frequency')

# By actual default status
new_data[new_data['default'] == 0]['default_probability'].hist(bins=30, alpha=0.6, label='No Default', ax=axes[1], color='green')
new_data[new_data['default'] == 1]['default_probability'].hist(bins=30, alpha=0.6, label='Default', ax=axes[1], color='red')
axes[1].set_title('Default Probabilities by Actual Status')
axes[1].set_xlabel('Default Probability')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
# Save the trained model
model.save('../../models/credit_risk/credit_risk_model.pkl')
print("Model saved successfully!")